In [48]:
import re

# Read original transliteration file
with open('./raw_data/quran_transliteration.txt', 'r', encoding='utf-8') as f:
    raw_text = f.read()

# Remove website header text
raw_text = raw_text.replace('The Calgary Islamic Homepage', '')

# Keep only letters, numbers, dots and spaces
clean_transliteration = re.sub(r'[^A-Za-z0-9. \n]+', '', raw_text)

# Save cleaned version
with open("./cleaned_data/cleaned.txt", "w", encoding="utf-8") as f:
    f.write(clean_transliteration)

In [49]:
# Split by "Surah"
surah_sections = clean_transliteration.split('Surah')

print("Total Surahs Found:", len(surah_sections))

# Save each surah as a separate file
for section in surah_sections[1:]:
    match = re.search(r'\d+', section)
    surah_number = match.group()

    with open(f'./cleaned_data/surahs/{surah_number}', 'w', encoding='utf-8') as f:
        f.write(section.strip())

Total Surahs Found: 115


In [50]:
import pandas as pd

# Load ayah dataset
df = pd.read_csv('./raw_data/ayahs.csv', header=None)

# Assign proper column names
df.columns = [
    'id', 'number', 'text', 'number_in_surah',
    'page', 'sura_id', 'hizb_id', 'juz_id',
    'sajda', 'created_at', 'updated_at'
]

df.columns = df.columns.str.strip()

In [51]:
import re

def normalize_arabic_for_buckwalter(text: str) -> str:
    """
    Normalize Arabic text:
    - Normalize alifs
    - Normalize hamza forms
    - Remove tashkeel
    - Remove tatweel
    - Keep only Arabic letters
    """

    if not isinstance(text, str):
        return text

    replacements = {
        'ٱ': 'ا',
        'ٰ': 'ا',
        'أ': 'ا',
        'إ': 'ا',
        'آ': 'ا',
        'ى': 'ي',
        'ؤ': 'و',
        'ئ': 'ي',
        'ة': 'ه',
    }

    for src, target in replacements.items():
        text = text.replace(src, target)

    # Remove diacritics and decorative marks
    diacritics_pattern = re.compile(
        r'[\u0610-\u061A\u064B-\u065F\u06D6-\u06ED\u0640]'
    )
    text = re.sub(diacritics_pattern, '', text)

    # Keep Arabic letters only
    text = re.sub(r'[^\u0621-\u064A\s]', '', text)

    # Normalize spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [52]:
# Remove Basmala phrase
df['text'] = df['text'].str.replace(
    r'بِسْمِ ٱللَّهِ ٱلرَّحْمَٰنِ ٱلرَّحِيمِ',
    '',
    regex=True
)

# Apply normalization
df['normalized_text'] = df['text'].apply(normalize_arabic_for_buckwalter)

df.head()

,id,number,text,number_in_surah,page,sura_id,hizb_id,juz_id,sajda,created_at,updated_at,normalized_text
0,1,1,,1,1,1,1,1,0,2018-06-07 08:06:54,2018-06-07 08:06:54,
1,2,2,ٱلْحَمْدُ لِلَّهِ رَبِّ ٱلْعَٰلَمِينَ,2,1,1,1,1,0,2018-06-07 08:06:54,2018-06-07 08:06:54,الحمد لله رب العالمين
2,3,3,ٱلرَّحْمَٰنِ ٱلرَّحِيمِ,3,1,1,1,1,0,2018-06-07 08:06:54,2018-06-07 08:06:54,الرحمان الرحيم
3,4,4,مَٰلِكِ يَوْمِ ٱلدِّينِ,4,1,1,1,1,0,2018-06-07 08:06:54,2018-06-07 08:06:54,مالك يوم الدين
4,5,5,إِيَّاكَ نَعْبُدُ وَإِيَّاكَ نَسْتَعِينُ,5,1,1,1,1,0,2018-06-07 08:06:54,2018-06-07 08:06:54,اياك نعبد واياك نستعين


In [53]:
import re

def clean_transliteration_line(text: str) -> str:
    if not isinstance(text, str):
        return text

    text = text.replace('\n', ' ')
    text = re.sub(r'\d+$', '', text)
    return text.strip()


def remove_quranic_stops(text: str) -> str:
    if not isinstance(text, str):
        return text

    text = re.sub(r'[\u06D6-\u06ED]', '', text)
    text = re.sub(r'[ۖۗۘۙۚۛۜ۝۞]', '', text)

    return text


def arabic_to_simple_latin(text: str) -> str:
    if not isinstance(text, str):
        return text

    mapping = {
        'ا': 'a','ب': 'b','ت': 't','ث': 'th','ج': 'j',
        'ح': 'h','خ': 'kh','د': 'd','ذ': 'dh','ر': 'r',
        'ز': 'z','س': 's','ش': 'sh','ص': 's','ض': 'd',
        'ط': 't','ظ': 'z','ع': 'a','غ': 'gh','ف': 'f',
        'ق': 'q','ك': 'k','ل': 'l','م': 'm','ن': 'n',
        'ه': 'h','و': 'w','ي': 'y','ء': 'a'
    }

    return ''.join(mapping.get(c, c) for c in text)

In [54]:
all_records = [
    ('بِسْمِ','بسم','Bismi','bsm'),
    ('ٱللَّهِ','الله','Allahi','allh'),
    ('ٱلرَّحْمَٰنِ','الرحمان','alrrahmani','alrhman'),
    ('ٱلرَّحِيمِ','الرحيم','alrraheemi','alrhym'),
]

for surah_id in range(1, 115):

    with open(f'cleaned_data/surahs/{surah_id}', 'r', encoding='utf-8') as f:
        surah_text = f.read()

    transliteration_verses = surah_text.split('.')

    for verse_index in range(2, len(transliteration_verses)):

        transliteration_verses[verse_index] = clean_transliteration_line(
            transliteration_verses[verse_index]
        )

        phonetic_words = transliteration_verses[verse_index].split()

        verse = df.loc[
            (df['number_in_surah'] == verse_index) &
            (df['sura_id'] == surah_id),
            ['text', 'normalized_text']
        ]

        if not verse.empty:

            arabic_words = remove_quranic_stops(
                verse['text'].iloc[0]
            ).split()

            normalized_words = verse['normalized_text'].iloc[0].split()

            simple_latin_words = [
                arabic_to_simple_latin(word)
                for word in normalized_words
            ]

            if (
                len(arabic_words) ==
                len(normalized_words) ==
                len(phonetic_words) ==
                len(simple_latin_words)
            ):
                records = [
                    (a, n, p, s)
                    for a, n, p, s in zip(
                        arabic_words,
                        normalized_words,
                        phonetic_words,
                        simple_latin_words
                    )
                ]

                all_records.extend(records)

        else:
            print(f"Verse {verse_index} in Surah {surah_id} not found.")

In [55]:
words_df = pd.DataFrame(
    all_records,
    columns=[
        'arabic_word',
        'normalized_arabic',
        'phonetic_word',
        'simple_latin'
    ]
)

words_df.to_csv('quran_words.csv', encoding='utf-8', index=False)

words_df.head(20)

,arabic_word,normalized_arabic,phonetic_word,simple_latin
0,بِسْمِ,بسم,Bismi,bsm
1,ٱللَّهِ,الله,Allahi,allh
2,ٱلرَّحْمَٰنِ,الرحمان,alrrahmani,alrhman
3,ٱلرَّحِيمِ,الرحيم,alrraheemi,alrhym
4,ٱلْحَمْدُ,الحمد,Alhamdu,alhmd
5,لِلَّهِ,لله,lillahi,llh
6,رَبِّ,رب,rabbi,rb
7,ٱلْعَٰلَمِينَ,العالمين,alAAalameena,alaalmyn
8,ٱلرَّحْمَٰنِ,الرحمان,Alrrahmani,alrhman
9,ٱلرَّحِيمِ,الرحيم,alrraheemi,alrhym
